# TFM — Detección de Anomalías en ECG con Big Data
## Capa Gold: Modelos de Machine Learning
**Andrea Ríos López — Master Big Data & Advanced Analytics**

Este notebook implementa los dos modelos de detección de anomalías sobre el dataset PTB-XL:
- **Parte A**: Isolation Forest (features estadísticas + frecuenciales)
- **Parte B**: LSTM Autoencoder (señal cruda temporal)

Al final se consolidan los scores en un DataFrame listo para exportar a Snowflake.

---
## PARTE A — Isolation Forest

### A.1 — Instalación de dependencias

In [ ]:
%pip install wfdb --quiet

### A.2 — Configuración: librerías y conexión S3

In [ ]:
import boto3
import wfdb
import numpy as np
import pandas as pd
import io
import os
import ast
import pickle
from scipy import stats
from scipy.fft import fft
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    precision_recall_curve,
    roc_curve,
    auc
)
import matplotlib.pyplot as plt

# Credenciales y parámetros S3
ACCESS_KEY = ''  # completar antes de ejecutar
SECRET_KEY = ''  # completar antes de ejecutar
BUCKET     = 'ptbxl-tfm-andrea'
REGION     = 'eu-west-1'
PREFIX     = 'bronze/FileYear=2026/FileMonth=04/FileDay=13'

s3 = boto3.client(
    's3',
    aws_access_key_id     = ACCESS_KEY,
    aws_secret_access_key = SECRET_KEY,
    region_name           = REGION
)

print('Conexión S3 OK')

### A.3 — Carga de metadatos

In [ ]:
def diagnostico_principal(scp_str):
    try:
        d = ast.literal_eval(scp_str)
        return max(d, key=d.get) if d else 'UNKNOWN'
    except:
        return 'UNKNOWN'

obj = s3.get_object(Bucket=BUCKET, Key=f'{PREFIX}/ptbxl_database.csv')
df_meta = pd.read_csv(io.BytesIO(obj['Body'].read()))
df_meta['diagnostico'] = df_meta['scp_codes'].apply(diagnostico_principal)

df_normal     = df_meta[df_meta['diagnostico'] == 'NORM'].reset_index(drop=True)
df_patologico = df_meta[df_meta['diagnostico'] != 'NORM'].reset_index(drop=True)

print(f'Registros normales:    {len(df_normal):,}')
print(f'Registros patologicos: {len(df_patologico):,}')

### A.4 — Extracción de features estadísticas y frecuenciales

In [ ]:
def extraer_features(ecg_id, filename_lr, s3_client, bucket, prefix):
    """
    Extrae features por derivación: media, desviación estándar, mínimo, máximo,
    rango, skewness, kurtosis y energía FFT.
    Devuelve un diccionario con ecg_id y todas las features, o 'error' si falla.
    """
    try:
        base_name  = filename_lr.split('/')[-1]
        subcarpeta = filename_lr.split('/')[-2]
        local_base = f'/tmp/{base_name}'

        for ext in ['.dat', '.hea']:
            key = f'{prefix}/records100/{subcarpeta}/{base_name}{ext}'
            s3_client.download_file(bucket, key, f'{local_base}{ext}')

        record = wfdb.rdrecord(local_base)
        signal = record.p_signal
        features = {'ecg_id': ecg_id}

        for i, lead in enumerate(record.sig_name):
            s = signal[:, i]
            fft_vals = np.abs(fft(s))
            features[f'{lead}_mean']       = np.mean(s)
            features[f'{lead}_std']        = np.std(s)
            features[f'{lead}_min']        = np.min(s)
            features[f'{lead}_max']        = np.max(s)
            features[f'{lead}_range']      = np.max(s) - np.min(s)
            features[f'{lead}_skew']       = stats.skew(s)
            features[f'{lead}_kurtosis']   = stats.kurtosis(s)
            features[f'{lead}_fft_energy'] = np.sum(fft_vals**2) / len(fft_vals)

        for ext in ['.dat', '.hea']:
            os.remove(f'{local_base}{ext}')

        return features

    except Exception as e:
        return {'ecg_id': ecg_id, 'error': str(e)}

### A.5 — Procesamiento de registros normales (con checkpoints en S3)

In [ ]:
# Si ya existe el archivo en Gold, saltar esta celda y ejecutar A.5b
features_list = []
errores       = []

for idx, row in df_normal.iterrows():
    result = extraer_features(
        ecg_id      = row['ecg_id'],
        filename_lr = row['filename_lr'],
        s3_client   = s3,
        bucket      = BUCKET,
        prefix      = PREFIX
    )
    if 'error' in result:
        errores.append(result)
    else:
        features_list.append(result)

    if len(features_list) % 500 == 0 and len(features_list) > 0:
        df_cp = pd.DataFrame(features_list)
        buf = io.BytesIO()
        df_cp.to_parquet(buf, index=False)
        buf.seek(0)
        s3.put_object(
            Bucket = BUCKET,
            Key    = f'gold/FileYear=2026/FileMonth=04/FileDay=13/features_normales_cp_{len(features_list)}.parquet',
            Body   = buf.getvalue()
        )
        print(f'Checkpoint: {len(features_list):,} registros — Errores: {len(errores)}')

df_features = pd.DataFrame(features_list)
buf = io.BytesIO()
df_features.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(
    Bucket = BUCKET,
    Key    = 'gold/FileYear=2026/FileMonth=04/FileDay=13/features_normales_completo.parquet',
    Body   = buf.getvalue()
)
print(f'Guardado en Gold: {len(df_features):,} registros x {len(df_features.columns)} columnas')
print(f'Errores totales:  {len(errores)}')

### A.5b — Cargar features normales desde Gold (si ya existen)

In [ ]:
obj = s3.get_object(
    Bucket = BUCKET,
    Key    = 'gold/FileYear=2026/FileMonth=04/FileDay=13/features_normales_completo.parquet'
)
df_features = pd.read_parquet(io.BytesIO(obj['Body'].read()))
print(f'Features normales cargadas: {len(df_features):,} registros x {len(df_features.columns)} columnas')

### A.6 — Entrenamiento del Isolation Forest

In [ ]:
X_train = df_features.drop(columns=['ecg_id']).values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

modelo_if = IsolationForest(
    n_estimators  = 100,
    contamination = 0.05,
    random_state  = 42,
    n_jobs        = -1
)
modelo_if.fit(X_scaled)

scores_train      = modelo_if.decision_function(X_scaled)
predicciones_train = modelo_if.predict(X_scaled)
n_outliers_train  = (predicciones_train == -1).sum()

print(f'Modelo entrenado sobre {len(X_train):,} registros normales')
print(f'Outliers en entrenamiento: {n_outliers_train} ({n_outliers_train/len(X_train)*100:.1f}%)')
print(f'Score medio:               {scores_train.mean():.4f}')

### A.7 — Guardado del modelo en S3

In [ ]:
model_bytes = pickle.dumps({'modelo': modelo_if, 'scaler': scaler})
s3.put_object(
    Bucket = BUCKET,
    Key    = 'gold/models/isolation_forest_v1.pkl',
    Body   = model_bytes
)
print('Modelo guardado en S3: gold/models/isolation_forest_v1.pkl')

### A.8 — Extracción de features patológicos y evaluación

In [ ]:
features_pat = []
errores_pat  = []

for idx, row in df_patologico.iterrows():
    result = extraer_features(
        ecg_id      = row['ecg_id'],
        filename_lr = row['filename_lr'],
        s3_client   = s3,
        bucket      = BUCKET,
        prefix      = PREFIX
    )
    if 'error' in result:
        errores_pat.append(result)
    else:
        features_pat.append(result)

    if len(features_pat) % 500 == 0 and len(features_pat) > 0:
        df_cp = pd.DataFrame(features_pat)
        buf = io.BytesIO()
        df_cp.to_parquet(buf, index=False)
        buf.seek(0)
        s3.put_object(
            Bucket = BUCKET,
            Key    = f'gold/FileYear=2026/FileMonth=04/FileDay=13/features_patologicos_cp_{len(features_pat)}.parquet',
            Body   = buf.getvalue()
        )
        print(f'Checkpoint: {len(features_pat):,} registros — Errores: {len(errores_pat)}')

df_pat = pd.DataFrame(features_pat)
buf = io.BytesIO()
df_pat.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(
    Bucket = BUCKET,
    Key    = 'gold/FileYear=2026/FileMonth=04/FileDay=13/features_patologicos_completo.parquet',
    Body   = buf.getvalue()
)
print(f'Guardado en Gold: {len(df_pat):,} registros patologicos')
print(f'Errores totales:  {len(errores_pat)}')

### A.8b — Cargar features patológicos desde Gold (si ya existen)

In [ ]:
obj = s3.get_object(
    Bucket = BUCKET,
    Key    = 'gold/FileYear=2026/FileMonth=04/FileDay=13/features_patologicos_completo.parquet'
)
df_pat = pd.read_parquet(io.BytesIO(obj['Body'].read()))
print(f'Features patologicas cargadas: {len(df_pat):,} registros x {len(df_pat.columns)} columnas')

### A.9 — Métricas de evaluación

In [ ]:
X_normal_scaled = scaler.transform(df_features.drop(columns=['ecg_id']).values)
X_pat_scaled    = scaler.transform(df_pat.drop(columns=['ecg_id']).values)

scores_normal = modelo_if.decision_function(X_normal_scaled)
scores_pat    = modelo_if.decision_function(X_pat_scaled)

y_true   = np.concatenate([np.zeros(len(scores_normal)), np.ones(len(scores_pat))])
y_scores = np.concatenate([-scores_normal, -scores_pat])

pred_normal = modelo_if.predict(X_normal_scaled)
pred_pat    = modelo_if.predict(X_pat_scaled)
y_pred = np.concatenate([
    (pred_normal == -1).astype(int),
    (pred_pat    == -1).astype(int)
])

auc_roc_if = roc_auc_score(y_true, y_scores)
precision, recall, _ = precision_recall_curve(y_true, y_scores)
auc_pr_if  = auc(recall, precision)

tp = ((y_pred == 1) & (y_true == 1)).sum()
fp = ((y_pred == 1) & (y_true == 0)).sum()

print('=' * 50)
print('EVALUACION — ISOLATION FOREST')
print('=' * 50)
print(f'AUC-ROC:              {auc_roc_if:.4f}')
print(f'AUC Precision-Recall: {auc_pr_if:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=['Normal', 'Patologico'], digits=4))
print('=' * 50)
print(f'Patologicos detectados: {tp:,} / {int(y_true.sum()):,} ({tp/y_true.sum()*100:.1f}%)')
print(f'Falsos positivos:       {fp:,} / {len(scores_normal):,} ({fp/len(scores_normal)*100:.1f}%)')

### A.10 — Curvas ROC y Precision-Recall

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_scores)
precision_c, recall_c, _ = precision_recall_curve(y_true, y_scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='#2E75B6', linewidth=2,
             label=f'AUC-ROC = {auc_roc_if:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatorio (AUC = 0.50)')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curva ROC — Isolation Forest')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(recall_c, precision_c, color='#E74C3C', linewidth=2,
             label=f'AUC-PR = {auc_pr_if:.4f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall — Isolation Forest')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
display(fig)

### A.11 — Análisis de detección por diagnóstico

In [ ]:
pred_pat_bin = (pred_pat == -1).astype(int)

deteccion_if = []
for i, row in df_patologico.iterrows():
    if i < len(pred_pat_bin):
        deteccion_if.append({
            'ecg_id'     : row['ecg_id'],
            'diagnostico': row['diagnostico'],
            'score_if'   : float(-scores_pat[i]),
            'detectado'  : int(pred_pat_bin[i])
        })

df_if_eval = pd.DataFrame(deteccion_if)

resumen_if = df_if_eval.groupby('diagnostico').agg(
    total      = ('ecg_id', 'count'),
    detectados = ('detectado', 'sum')
).reset_index()
resumen_if['tasa_deteccion'] = (
    resumen_if['detectados'] / resumen_if['total'] * 100
).round(1)
resumen_if = resumen_if.sort_values('tasa_deteccion', ascending=False)

display(resumen_if)

---
## PARTE B — LSTM Autoencoder

### B.1 — Instalación de dependencias

In [ ]:
%pip install torch torchvision --quiet

### B.2 — Librerías adicionales para LSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

### B.3 — Función para leer señal cruda desde S3

In [ ]:
def leer_señal(filename_lr, s3_client, bucket, prefix, derivacion=0, n_puntos=1000):
    """
    Lee una señal ECG desde S3 y devuelve los primeros n_puntos
    de la derivación indicada (0 = derivación I por defecto).
    La señal se normaliza entre -1 y 1.
    """
    try:
        base_name  = filename_lr.split('/')[-1]
        subcarpeta = filename_lr.split('/')[-2]
        local_base = f'/tmp/lstm_{base_name}'

        for ext in ['.dat', '.hea']:
            key = f'{prefix}/records100/{subcarpeta}/{base_name}{ext}'
            s3_client.download_file(bucket, key, f'{local_base}{ext}')

        record = wfdb.rdrecord(local_base)
        signal = record.p_signal[:n_puntos, derivacion]

        for ext in ['.dat', '.hea']:
            os.remove(f'{local_base}{ext}')

        s_min, s_max = signal.min(), signal.max()
        if s_max - s_min > 0:
            signal = 2 * (signal - s_min) / (s_max - s_min) - 1

        return signal.astype(np.float32)

    except:
        return None

print('Funcion leer_señal definida OK')

### B.4 — Carga de señales normales para entrenamiento

In [ ]:
# Separar pliegues: 1-8 entrenamiento, 9 validación, 10 test
df_train = df_normal[df_normal['strat_fold'] <= 8].reset_index(drop=True)
df_val   = df_normal[df_normal['strat_fold'] == 9].reset_index(drop=True)

print(f'Registros entrenamiento: {len(df_train):,}')
print(f'Registros validacion:    {len(df_val):,}')
print('Cargando señales...')

señales_train = []
for idx, row in df_train.iterrows():
    s = leer_señal(row['filename_lr'], s3, BUCKET, PREFIX)
    if s is not None:
        señales_train.append(s)
    if idx % 500 == 0:
        print(f'  {idx}/{len(df_train)} — cargadas: {len(señales_train)}')

señales_val = []
for idx, row in df_val.iterrows():
    s = leer_señal(row['filename_lr'], s3, BUCKET, PREFIX)
    if s is not None:
        señales_val.append(s)

X_train_lstm = np.array(señales_train)
X_val_lstm   = np.array(señales_val)

print(f'X_train shape: {X_train_lstm.shape}')
print(f'X_val shape:   {X_val_lstm.shape}')

# Guardar en S3 para no recargar
for key, arr in [
    ('gold/lstm/X_train_señales.npy', X_train_lstm),
    ('gold/lstm/X_val_señales.npy',   X_val_lstm)
]:
    buf = io.BytesIO()
    np.save(buf, arr)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
    print(f'Guardado: {key}')

### B.4b — Cargar señales normales desde Gold (si ya existen)

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key='gold/lstm/X_train_señales.npy')
X_train_lstm = np.load(io.BytesIO(obj['Body'].read()))

obj = s3.get_object(Bucket=BUCKET, Key='gold/lstm/X_val_señales.npy')
X_val_lstm = np.load(io.BytesIO(obj['Body'].read()))

print(f'X_train shape: {X_train_lstm.shape}')
print(f'X_val shape:   {X_val_lstm.shape}')

### B.5 — Arquitectura LSTM Autoencoder

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, latent_size=32, num_layers=2):
        super(LSTMAutoencoder, self).__init__()
        self.encoder = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True, dropout=0.2
        )
        self.latent        = nn.Linear(hidden_size, latent_size)
        self.decoder_input = nn.Linear(latent_size, hidden_size)
        self.decoder = nn.LSTM(
            input_size=hidden_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True, dropout=0.2
        )
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        _, (hidden, _) = self.encoder(x)
        encoded        = self.latent(hidden[-1])
        dec_input      = self.decoder_input(encoded)
        dec_input      = dec_input.unsqueeze(1).repeat(1, seq_len, 1)
        decoded, _     = self.decoder(dec_input)
        output         = self.output_layer(decoded)
        return output, encoded

modelo_lstm = LSTMAutoencoder().to(device)
total_params = sum(p.numel() for p in modelo_lstm.parameters())
print(f'Arquitectura lista — parametros totales: {total_params:,}')

### B.6 — Entrenamiento del LSTM Autoencoder

In [ ]:
EPOCHS     = 20
BATCH_SIZE = 64
LR         = 0.001

tensor_train = torch.FloatTensor(X_train_lstm).unsqueeze(-1)
tensor_val   = torch.FloatTensor(X_val_lstm).unsqueeze(-1)

loader_train = DataLoader(TensorDataset(tensor_train), batch_size=BATCH_SIZE, shuffle=True)
loader_val   = DataLoader(TensorDataset(tensor_val),   batch_size=BATCH_SIZE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(modelo_lstm.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    modelo_lstm.train()
    loss_train = 0.0
    for (batch,) in loader_train:
        batch = batch.to(device)
        output, _ = modelo_lstm(batch)
        loss = criterion(output, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_train += loss.item()

    modelo_lstm.eval()
    loss_val = 0.0
    with torch.no_grad():
        for (batch,) in loader_val:
            batch = batch.to(device)
            output, _ = modelo_lstm(batch)
            loss_val += criterion(output, batch).item()

    print(f'Epoch {epoch:02d}/{EPOCHS} — '
          f'Loss train: {loss_train/len(loader_train):.6f} | '
          f'Loss val:   {loss_val/len(loader_val):.6f}')

print('Entrenamiento finalizado')

### B.7 — Guardado del modelo LSTM en S3

In [ ]:
model_bytes = pickle.dumps({
    'modelo_state': modelo_lstm.state_dict(),
    'epochs'      : EPOCHS,
    'hidden_size' : 64,
    'latent_size' : 32,
    'num_layers'  : 2
})
s3.put_object(
    Bucket = BUCKET,
    Key    = 'gold/models/lstm_autoencoder_v1.pkl',
    Body   = model_bytes
)
print('Modelo guardado en S3: gold/models/lstm_autoencoder_v1.pkl')

### B.8 — Función de cálculo de error de reconstrucción

In [ ]:
def calcular_errores(señales_array, modelo, batch_size=64):
    """
    Calcula el error de reconstrucción (MSE) por señal.
    Un error alto indica que la señal es anómala respecto al patron aprendido.
    """
    errores = []
    modelo.eval()
    with torch.no_grad():
        for i in range(0, len(señales_array), batch_size):
            batch  = torch.FloatTensor(señales_array[i:i+batch_size]).unsqueeze(-1).to(device)
            output, _ = modelo(batch)
            mse    = ((output - batch) ** 2).mean(dim=(1, 2))
            errores.extend(mse.cpu().numpy().tolist())
    return np.array(errores)

### B.9 — Evaluación sobre señales patológicas

In [ ]:
print('Calculando errores sobre normales de entrenamiento...')
errores_normales = calcular_errores(X_train_lstm, modelo_lstm)
print(f'Error medio normales: {errores_normales.mean():.6f}')

print('Cargando señales patologicas...')
señales_pat  = []
errores_carga = []
for idx, row in df_patologico.iterrows():
    s = leer_señal(row['filename_lr'], s3, BUCKET, PREFIX)
    if s is not None:
        señales_pat.append(s)
    else:
        errores_carga.append(row['ecg_id'])
    if idx % 1000 == 0:
        print(f'  {idx}/{len(df_patologico)} — cargadas: {len(señales_pat)}')

X_pat_señales = np.array(señales_pat)
print(f'Señales patologicas cargadas: {X_pat_señales.shape}')

print('Calculando errores sobre patologicos...')
errores_pat_lstm = calcular_errores(X_pat_señales, modelo_lstm)
print(f'Error medio patologicos: {errores_pat_lstm.mean():.6f}')

# Guardar señales patológicas en S3
buf = io.BytesIO()
np.save(buf, X_pat_señales)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key='gold/lstm/X_pat_señales.npy', Body=buf.getvalue())
print('Señales patologicas guardadas en S3')

### B.10 — Métricas LSTM Autoencoder

In [ ]:
umbral_lstm = np.percentile(errores_normales, 95)

y_true_lstm   = np.concatenate([np.zeros(len(errores_normales)), np.ones(len(errores_pat_lstm))])
y_scores_lstm = np.concatenate([errores_normales, errores_pat_lstm])
y_pred_lstm   = (y_scores_lstm > umbral_lstm).astype(int)

auc_roc_lstm = roc_auc_score(y_true_lstm, y_scores_lstm)
prec, rec, _ = precision_recall_curve(y_true_lstm, y_scores_lstm)
auc_pr_lstm  = auc(rec, prec)

print('=' * 50)
print('EVALUACION — LSTM AUTOENCODER')
print('=' * 50)
print(f'Umbral (percentil 95 normales): {umbral_lstm:.6f}')
print(f'Error medio normales:           {errores_normales.mean():.6f}')
print(f'Error medio patologicos:        {errores_pat_lstm.mean():.6f}')
print(f'AUC-ROC:                        {auc_roc_lstm:.4f}')
print(f'AUC Precision-Recall:           {auc_pr_lstm:.4f}')
print()
print(classification_report(y_true_lstm, y_pred_lstm, target_names=['Normal', 'Patologico'], digits=4))
print('=' * 50)

### B.11 — Análisis de detección por diagnóstico (LSTM)

In [ ]:
deteccion_lstm = []
for idx, row in df_patologico.iterrows():
    if idx < len(errores_pat_lstm):
        deteccion_lstm.append({
            'ecg_id'     : row['ecg_id'],
            'diagnostico': row['diagnostico'],
            'error_lstm' : float(errores_pat_lstm[idx]),
            'detectado'  : int(errores_pat_lstm[idx] > umbral_lstm)
        })

df_lstm_eval = pd.DataFrame(deteccion_lstm)

resumen_lstm = df_lstm_eval.groupby('diagnostico').agg(
    total      = ('ecg_id', 'count'),
    detectados = ('detectado', 'sum')
).reset_index()
resumen_lstm['tasa_deteccion'] = (
    resumen_lstm['detectados'] / resumen_lstm['total'] * 100
).round(1)

display(resumen_lstm.sort_values('tasa_deteccion', ascending=False))

---
## PARTE C — Consolidación de resultados para Snowflake

### C.1 — Construcción del DataFrame `anomaly_scores`

Este DataFrame corresponde a la tabla `anomaly_scores` definida en la arquitectura Snowflake del TFM.
Contiene un registro por modelo y por ECG evaluado, con el score, la etiqueta binaria y la fecha de inferencia.

In [ ]:
from datetime import date

INFERENCE_DATE = str(date.today())
THRESHOLD_IF   = float(np.percentile(-scores_normal, 95))
THRESHOLD_LSTM = float(umbral_lstm)

# Isolation Forest — registros patologicos evaluados
rows_if = []
for i, row in df_patologico.iterrows():
    if i < len(scores_pat):
        rows_if.append({
            'ecg_id'        : int(row['ecg_id']),
            'model_name'    : 'isolation_forest_v1',
            'anomaly_score' : float(-scores_pat[i]),
            'is_anomaly'    : bool(pred_pat[i] == -1),
            'threshold'     : THRESHOLD_IF,
            'inference_date': INFERENCE_DATE
        })

# LSTM Autoencoder — registros patologicos evaluados
rows_lstm = []
for i, row in df_patologico.iterrows():
    if i < len(errores_pat_lstm):
        rows_lstm.append({
            'ecg_id'        : int(row['ecg_id']),
            'model_name'    : 'lstm_autoencoder_v1',
            'anomaly_score' : float(errores_pat_lstm[i]),
            'is_anomaly'    : bool(errores_pat_lstm[i] > THRESHOLD_LSTM),
            'threshold'     : THRESHOLD_LSTM,
            'inference_date': INFERENCE_DATE
        })

df_anomaly_scores = pd.concat([
    pd.DataFrame(rows_if),
    pd.DataFrame(rows_lstm)
], ignore_index=True)

print(f'Registros totales en anomaly_scores: {len(df_anomaly_scores):,}')
print(f'  - Isolation Forest: {len(rows_if):,}')
print(f'  - LSTM Autoencoder: {len(rows_lstm):,}')
display(df_anomaly_scores.head(10))

### C.2 — Guardado de `anomaly_scores` en S3 (staging para Snowflake)

In [ ]:
buf = io.BytesIO()
df_anomaly_scores.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(
    Bucket = BUCKET,
    Key    = f'gold/snowflake/anomaly_scores_{INFERENCE_DATE}.parquet',
    Body   = buf.getvalue()
)
print(f'anomaly_scores guardado en S3: gold/snowflake/anomaly_scores_{INFERENCE_DATE}.parquet')
print(f'Listo para cargar en Snowflake')

### C.3 — Registro del modelo en `model_registry`

In [ ]:
df_model_registry = pd.DataFrame([
    {
        'model_name' : 'isolation_forest_v1',
        'version'    : 'v1',
        'train_date' : INFERENCE_DATE,
        'auc_roc'    : round(float(auc_roc_if),  4),
        'auc_pr'     : round(float(auc_pr_if),   4),
        'n_records'  : len(df_features)
    },
    {
        'model_name' : 'lstm_autoencoder_v1',
        'version'    : 'v1',
        'train_date' : INFERENCE_DATE,
        'auc_roc'    : round(float(auc_roc_lstm), 4),
        'auc_pr'     : round(float(auc_pr_lstm),  4),
        'n_records'  : len(X_train_lstm)
    }
])

display(df_model_registry)

buf = io.BytesIO()
df_model_registry.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(
    Bucket = BUCKET,
    Key    = 'gold/snowflake/model_registry.parquet',
    Body   = buf.getvalue()
)
print('model_registry guardado en S3')